# AIC2024 KIS — Phase 2: CLIP embeddings theo từng clip (Kaggle)

Notebook **self-contained**. Mỗi video sinh một shard riêng để chạy Kaggle theo nhiều batch.

## Chuẩn bị
1. Zip `artifacts/` (`metadata.sqlite` + `keyframes/`) và upload thành Kaggle Dataset.
2. Notebook Settings: bật **GPU** và **Internet**.
3. Add dataset, chọn batch (bên dưới), chạy hết cell.
4. Tải `artifacts_embed_shards.zip`, giải nén vào `artifacts/artifacts_embed/`
   → `artifacts/artifacts_embed/shards/<video_id>/...`
5. Ở local chạy `uv run ir merge` để build combined embeddings + FAISS.

## Chọn batch
- Mặc định: tất cả video trong DB.
- Theo lô: `EMBED_VIDEO_OFFSET` + `EMBED_VIDEO_LIMIT` (ví dụ 0/100, 100/100, ...).
- Hoặc chỉ định: `EMBED_VIDEO_IDS=L01_V001,L01_V002`.
- Nếu dataset chỉ chứa keyframe của một số video, dùng `EMBED_VIDEO_IDS` đúng danh sách đó.
- Resume: shard đã xong trong `/kaggle/working` sẽ được bỏ qua.

## Output mỗi shard
`shards/<video_id>/embeddings.f16.npy`, `id_map.npy`, `manifest.json`.


## 1. Cấu hình


In [ ]:
import os
from pathlib import Path

MODEL_NAME = "sentence-transformers/clip-ViT-B-16"
DIM = 512
BATCH_SIZE = 128

ARTIFACTS_DIR = os.environ.get("EMBED_ARTIFACTS_DIR") or None
WORK_DIR = Path(os.environ.get("EMBED_WORK_DIR", "/kaggle/working/embed"))
SHARDS_DIR = Path(os.environ.get("EMBED_SHARDS_DIR", str(WORK_DIR / "shards")))
ZIP_PATH = Path(os.environ.get("EMBED_ZIP_PATH", "/kaggle/working/artifacts_embed_shards.zip"))
RESUME = os.environ.get("EMBED_RESUME", "1") != "0"

VIDEO_IDS = [
    value.strip()
    for value in os.environ.get("EMBED_VIDEO_IDS", "").split(",")
    if value.strip()
]
VIDEO_OFFSET = int(os.environ.get("EMBED_VIDEO_OFFSET", "0"))
VIDEO_LIMIT = int(os.environ.get("EMBED_VIDEO_LIMIT", "0"))

print("model:", MODEL_NAME, "| dim:", DIM, "| batch:", BATCH_SIZE)
print("artifacts override:", ARTIFACTS_DIR)
print("work:", WORK_DIR)
print("shards:", SHARDS_DIR)
print("zip:", ZIP_PATH)
print("video_ids:", VIDEO_IDS or "(all)", "| offset:", VIDEO_OFFSET, "| limit:", VIDEO_LIMIT)


## 2. Cài dependency

Kaggle đã có `torch`; cell này bổ sung `sentence-transformers` và `faiss-cpu`.


In [ ]:
import subprocess
import sys

if os.environ.get("EMBED_SKIP_INSTALL") != "1":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "faiss-cpu"],
        check=True,
    )
else:
    print("skip install (EMBED_SKIP_INSTALL=1)")


## 3. Tìm `artifacts/` và copy DB

`/kaggle/input` là read-only, còn SQLite ở WAL mode cần quyền ghi file `-shm`,
nên notebook copy DB sang `/kaggle/working` rồi mở bản copy.


In [ ]:
import shutil
import sqlite3

PATTERNS = (
    "*/artifacts/metadata.sqlite",
    "*/metadata.sqlite",
    "*/*/artifacts/metadata.sqlite",
    "*/*/metadata.sqlite",
    "*/*/*/artifacts/metadata.sqlite",
)


def candidate_dirs() -> list[Path]:
    dirs: list[Path] = []
    if ARTIFACTS_DIR:
        dirs.append(Path(ARTIFACTS_DIR))
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for pattern in PATTERNS:
            dirs.extend(db.parent for db in input_root.glob(pattern))
        if not dirs:
            dirs.extend(db.parent for db in input_root.rglob("metadata.sqlite"))
    local = Path.cwd() / "artifacts"
    if local.is_dir():
        dirs.append(local)
    return dirs


candidates = []
seen = set()
for path in candidate_dirs():
    resolved = path.resolve()
    if resolved in seen:
        continue
    seen.add(resolved)
    if (resolved / "metadata.sqlite").is_file() and (resolved / "keyframes").is_dir():
        candidates.append(resolved)

if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy metadata.sqlite + keyframes/. "
        "Set EMBED_ARTIFACTS_DIR hoặc kiểm tra lại Kaggle dataset."
    )

ARTIFACTS_DIR = candidates[0]
SOURCE_DB_PATH = ARTIFACTS_DIR / "metadata.sqlite"

WORK_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = WORK_DIR / "metadata.sqlite"
for suffix in ("", "-wal"):
    source = Path(str(SOURCE_DB_PATH) + suffix)
    target = Path(str(DB_PATH) + suffix)
    if source.is_file():
        shutil.copy2(source, target)
    elif suffix:
        target.unlink(missing_ok=True)
Path(str(DB_PATH) + "-shm").unlink(missing_ok=True)

print("artifacts:", ARTIFACTS_DIR)
print("db source:", SOURCE_DB_PATH)
print("db working copy:", DB_PATH)
if len(candidates) > 1:
    print("bỏ qua candidate khác:", [str(p) for p in candidates[1:]])

conn = sqlite3.connect(DB_PATH)
try:
    videos = conn.execute(
        "SELECT video_id, generation, duration_ms FROM videos ORDER BY video_id"
    ).fetchall()
finally:
    conn.close()
print(f"videos trong DB: {len(videos):,}")


## 4. Chọn batch video

Thứ tự video lấy theo `video_id` đã sort. `EMBED_VIDEO_OFFSET`/`EMBED_VIDEO_LIMIT`
cho phép chạy lô 100 video mỗi lần mà không cần sửa notebook.


In [ ]:
import numpy as np

conn = sqlite3.connect(DB_PATH)
try:
    rows = conn.execute(
        """
        SELECT k.kf_uid, k.video_id, k.generation, k.shot_idx, k.kf_idx, k.ts_ms, k.path
        FROM keyframes k
        JOIN videos v ON v.video_id = k.video_id AND k.generation = v.generation
        ORDER BY k.video_id, k.kf_uid
        """
    ).fetchall()
finally:
    conn.close()

by_video: dict[str, list] = {}
for row in rows:
    by_video.setdefault(row[1], []).append(row)

all_video_ids = sorted(by_video)
if VIDEO_IDS:
    selected = list(dict.fromkeys(VIDEO_IDS))
    missing_videos = [video_id for video_id in selected if video_id not in by_video]
    if missing_videos:
        raise ValueError(f"EMBED_VIDEO_IDS không có trong DB: {missing_videos}")
elif VIDEO_LIMIT > 0:
    selected = all_video_ids[VIDEO_OFFSET : VIDEO_OFFSET + VIDEO_LIMIT]
else:
    selected = all_video_ids[VIDEO_OFFSET:]

if not selected:
    raise RuntimeError("Không có video nào được chọn.")

missing_files = []
total_selected = 0
for video_id in selected:
    video_rows = by_video[video_id]
    total_selected += len(video_rows)
    for row in video_rows:
        image_path = ARTIFACTS_DIR / row[6]
        try:
            if image_path.stat().st_size == 0:
                missing_files.append(str(image_path))
        except OSError:
            missing_files.append(str(image_path))

print(f"videos in DB: {len(all_video_ids):,}")
print(f"selected videos: {len(selected):,} | keyframes: {total_selected:,}")
print(f"batch: offset={VIDEO_OFFSET} limit={VIDEO_LIMIT} explicit={len(VIDEO_IDS)}")
if selected:
    print("first:", selected[0], "| last:", selected[-1])
if not VIDEO_IDS and VIDEO_LIMIT > 0:
    print("next EMBED_VIDEO_OFFSET:", VIDEO_OFFSET + len(selected))
print(f"missing/empty: {len(missing_files):,}")
if missing_files:
    raise FileNotFoundError("Thiếu file keyframe:\n" + "\n".join(missing_files[:20]))


## 5. Load CLIP ViT-B/16


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("device:", DEVICE, "|", torch.cuda.get_device_name(0))
else:
    print("device:", DEVICE)

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model_dim = (
    model.get_embedding_dimension()
    if hasattr(model, "get_embedding_dimension")
    else model.get_sentence_embedding_dimension()
)
if model_dim != DIM:
    raise RuntimeError(f"model dim {model_dim} != configured {DIM}")
print("embedding dim:", model_dim)


## 6. Embed từng video thành shard

Mỗi video ghi vào `SHARDS_DIR/<video_id>/`. Shard đã xong được bỏ qua khi resume;
shard dở dang đọc `progress.json` để chạy tiếp.


In [ ]:
import hashlib
import json
import time
from datetime import UTC, datetime

from PIL import Image


def write_json(path: Path, payload: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    tmp = path.with_name(f"{path.stem}.tmp.npy")
    np.save(tmp, array)
    os.replace(tmp, path)


shard_summaries = []
overall_started = time.time()

for position, video_id in enumerate(selected, start=1):
    video_rows = by_video[video_id]
    total = len(video_rows)
    uids = np.asarray([row[0] for row in video_rows], dtype=np.int64)
    current_hash = hashlib.sha256(
        MODEL_NAME.encode() + b"\n" + uids.tobytes()
    ).hexdigest()

    shard_dir = SHARDS_DIR / video_id
    if not RESUME:
        shutil.rmtree(shard_dir, ignore_errors=True)
    shard_dir.mkdir(parents=True, exist_ok=True)

    embeddings_path = shard_dir / "embeddings.f16.npy"
    progress_path = shard_dir / "progress.json"
    plan_path = shard_dir / "plan.json"
    manifest_path = shard_dir / "manifest.json"

    rows_done = 0
    plan_ok = False
    if RESUME and embeddings_path.is_file() and plan_path.is_file():
        try:
            plan = json.loads(plan_path.read_text(encoding="utf-8"))
            probe = np.load(embeddings_path, mmap_mode="r")
            plan_ok = (
                plan.get("plan_hash") == current_hash
                and int(plan.get("count", -1)) == total
                and probe.shape == (total, DIM)
            )
            if plan_ok:
                progress = json.loads(progress_path.read_text(encoding="utf-8"))
                rows_done = min(max(0, int(progress.get("rows_done", 0))), total)
            del probe
        except (OSError, ValueError, KeyError):
            plan_ok = False

    if not plan_ok:
        embeddings_path.unlink(missing_ok=True)
        progress_path.unlink(missing_ok=True)
        plan_path.unlink(missing_ok=True)
        memmap = np.lib.format.open_memmap(
            str(embeddings_path), mode="w+", dtype=np.float16, shape=(total, DIM)
        )
        del memmap
        rows_done = 0

    write_json(
        plan_path,
        {
            "video_id": video_id,
            "model": MODEL_NAME,
            "dim": DIM,
            "count": total,
            "plan_hash": current_hash,
        },
    )

    if RESUME and rows_done == total and manifest_path.is_file():
        print(f"[{position}/{len(selected)}] {video_id}: skip (done)")
        shard_summaries.append({"video_id": video_id, "count": total, "skipped": True})
        continue

    write_json(progress_path, {"rows_done": rows_done, "status": "running"})
    started = time.time()
    memmap = np.lib.format.open_memmap(str(embeddings_path), mode="r+")
    for start in range(rows_done, total, BATCH_SIZE):
        chunk = video_rows[start : start + BATCH_SIZE]
        images = []
        for row in chunk:
            with Image.open(ARTIFACTS_DIR / row[6]) as image:
                images.append(image.convert("RGB"))

        vectors = model.encode(
            images,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        vectors = np.asarray(vectors, dtype=np.float32)
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        vectors = vectors / np.maximum(norms, 1e-12)
        memmap[start : start + len(chunk)] = vectors.astype(np.float16)
        memmap.flush()
        write_json(
            progress_path,
            {"rows_done": start + len(chunk), "status": "running"},
        )

    memmap.flush()
    del memmap
    write_json(progress_path, {"rows_done": total, "status": "done"})

    atomic_save_npy(shard_dir / "id_map.npy", uids)
    manifest = {
        "version": 1,
        "video_id": video_id,
        "created_at": datetime.now(UTC).isoformat(),
        "model": MODEL_NAME,
        "dim": DIM,
        "count": total,
        "dtype": "float16",
        "normalized": "l2",
        "order": "kf_uid",
        "plan_hash": current_hash,
        "files": {"embeddings": "embeddings.f16.npy", "id_map": "id_map.npy"},
    }
    write_json(manifest_path, manifest)
    shard_summaries.append({"video_id": video_id, "count": total, "skipped": False})
    print(
        f"[{position}/{len(selected)}] {video_id}: {total:,} keyframes "
        f"in {time.time() - started:,.1f}s"
    )

print(
    f"done {len(shard_summaries):,} videos in "
    f"{(time.time() - overall_started) / 60:,.1f} min"
)


## 7. Đóng gói shard

Zip chỉ chứa shard của batch này (không chứa keyframes, không chứa file tạm).


In [ ]:
import zipfile

README = """AIC2024 KIS - per-clip embedding shards
Extract into artifacts/artifacts_embed/ so paths become:
  artifacts/artifacts_embed/shards/<video_id>/{embeddings.f16.npy,id_map.npy,manifest.json}
Then run locally:
  uv run ir merge
"""

ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
written = 0
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as archive:
    for summary in shard_summaries:
        shard_dir = SHARDS_DIR / summary["video_id"]
        for path in sorted(shard_dir.rglob("*")):
            if not path.is_file() or path.name in {"progress.json", "plan.json"}:
                continue
            archive.write(
                path,
                arcname=path.relative_to(SHARDS_DIR.parent).as_posix(),
            )
            written += 1
    archive.writestr("README.txt", README)

print(f"zip: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:,.1f} MB, {written} files)")


## 8. Kiểm tra batch

Chỉ verify shape/count của từng shard; FAISS tổng được build ở local bằng `ir merge`.


In [ ]:
total_keyframes = 0
for summary in shard_summaries:
    shard_dir = SHARDS_DIR / summary["video_id"]
    manifest = json.loads((shard_dir / "manifest.json").read_text(encoding="utf-8"))
    embeddings = np.load(shard_dir / "embeddings.f16.npy", mmap_mode="r")
    ids = np.load(shard_dir / "id_map.npy")
    count = int(manifest["count"])
    assert embeddings.shape == (count, DIM), (summary["video_id"], embeddings.shape)
    assert ids.shape == (count,), (summary["video_id"], ids.shape)
    total_keyframes += count
    print(f"{summary['video_id']}: {count:,} keyframes")

print(f"batch: {len(shard_summaries):,} videos, {total_keyframes:,} keyframes")
if not VIDEO_IDS and VIDEO_LIMIT > 0:
    print("next EMBED_VIDEO_OFFSET:", VIDEO_OFFSET + len(selected))


## Sau khi chạy

1. Tải `artifacts_embed_shards.zip`.
2. Giải nén vào `artifacts/artifacts_embed/` (để có `artifacts/artifacts_embed/shards/...`).
3. Ở local chạy `uv run ir merge` — build `embeddings.f16.npy`, `id_map.npy`, `faiss.index`, `manifest.json` từ toàn bộ shard hiện có.
4. Có thể merge sau mỗi batch; `ir merge` luôn đọc lại tất cả shard.
